# Semantic Retrieval for arXiv Papers

This notebook builds a dense retrieval system over arXiv metadata. The target is `MRR@5 > 0.91` on a held-out query set where each query maps to one relevant paper.

## Stage 1. Exploratory Data Analysis

The EDA stage validates corpus size, missing values, duplicate IDs, query coverage, and text length distributions. The retrieval index uses `id`, `title`, and `abstract`.

In [ ]:
import os
import re
import time
import warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_colwidth", 140)

def find_data_dir() -> Path:
    '''Find the folder with project data regardless of where the notebook is opened.'''
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd / "nlp_s3_project",
        cwd.parent / "nlp_s3_project",
    ]
    for candidate in candidates:
        if (candidate / "arxiv-metadata-s.json").exists() and (candidate / "test_sample.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find arxiv-metadata-s.json and test_sample.csv. "
        "Place the files next to the notebook or inside an nlp_s3_project subfolder."
    )

DATA_DIR = find_data_dir()
METADATA_PATH = DATA_DIR / "arxiv-metadata-s.json"
TEST_PATH = DATA_DIR / "test_sample.csv"

print(f"DATA_DIR: {DATA_DIR}")
print(f"metadata: {METADATA_PATH.name}, {METADATA_PATH.stat().st_size / 1024**2:.1f} MB")
print(f"test:     {TEST_PATH.name}, {TEST_PATH.stat().st_size / 1024**2:.1f} MB")

In [ ]:
docs_raw = pd.read_json(METADATA_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("Corpus shape:", docs_raw.shape)
print("Test set shape:", test_raw.shape)

display(docs_raw[["id", "title", "abstract"]].head(3))
display(test_raw.head(3))

In [ ]:
def clean_text(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

docs = docs_raw.copy()
test = test_raw.copy()

docs["id"] = docs["id"].astype(str)
docs["title_clean"] = clean_text(docs["title"])
docs["abstract_clean"] = clean_text(docs["abstract"])
docs["text_for_search"] = (docs["title_clean"] + ". " + docs["abstract_clean"]).str.strip()

test["id"] = test["id"].astype(str)
test["query_clean"] = clean_text(test["query"])
test["abstract_clean"] = clean_text(test["abstract"])

quality_checks = pd.DataFrame(
    {
        "metric": [
            "documents",
            "test_queries",
            "unique_document_ids",
            "unique_test_ids",
            "test_ids_found_in_corpus",
            "empty_titles",
            "empty_abstracts",
            "duplicate_document_ids",
        ],
        "value": [
            len(docs),
            len(test),
            docs["id"].nunique(),
            test["id"].nunique(),
            test["id"].isin(set(docs["id"])).sum(),
            (docs["title_clean"] == "").sum(),
            (docs["abstract_clean"] == "").sum(),
            docs["id"].duplicated().sum(),
        ],
    }
)
display(quality_checks)

In [ ]:
docs["title_words"] = docs["title_clean"].str.split().str.len()
docs["abstract_words"] = docs["abstract_clean"].str.split().str.len()
test["query_words"] = test["query_clean"].str.split().str.len()

text_stats = pd.DataFrame(
    {
        "title_words": docs["title_words"].describe(),
        "abstract_words": docs["abstract_words"].describe(),
        "query_words": test["query_words"].describe(),
    }
)
display(text_stats.round(2))

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(docs["title_words"], bins=40, ax=axes[0], color="#2a9d8f")
axes[0].set_title("Title length, words")
axes[0].set_xlabel("Words")

sns.histplot(docs["abstract_words"], bins=50, ax=axes[1], color="#457b9d")
axes[1].set_title("Abstract length, words")
axes[1].set_xlabel("Words")

sns.histplot(test["query_words"], bins=30, ax=axes[2], color="#e76f51")
axes[2].set_title("Query length, words")
axes[2].set_xlabel("Words")

plt.tight_layout()
plt.show()

In [ ]:
if "categories" in docs.columns:
    first_category = docs["categories"].fillna("").str.split().str[0].replace("", np.nan)
    top_categories = first_category.value_counts().head(15).reset_index()
    top_categories.columns = ["category", "documents"]

    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_categories, x="documents", y="category", color="#457b9d")
    plt.title("Top 15 first arXiv categories")
    plt.xlabel("Documents")
    plt.ylabel("Category")
    plt.tight_layout()
    plt.show()

    display(top_categories)

### EDA Takeaways and Modeling Choice

The corpus contains about 98k documents and all test targets are present. Since queries are natural-language questions rather than exact keyword matches, dense retrieval is a better fit than lexical search.

## Stage 2. Retrieval System

The implementation uses `BAAI/bge-large-en-v1.5` for embeddings, cosine similarity via normalized vectors, a FAISS inner-product index, and an on-disk embedding cache for repeatable experiments.

In [ ]:
def mean_reciprocal_rank_at_k(predicted_ids, true_ids, k: int = 5):
    '''Compute MRR@k for one correct answer per query.'''
    reciprocal_ranks = []
    true_ids = [str(x) for x in true_ids]

    for candidates, true_id in zip(predicted_ids, true_ids):
        rank = 0
        for position, candidate_id in enumerate(candidates[:k], start=1):
            if str(candidate_id) == true_id:
                rank = position
                break
        reciprocal_ranks.append(0.0 if rank == 0 else 1.0 / rank)

    return float(np.mean(reciprocal_ranks)), np.array(reciprocal_ranks, dtype=np.float32)


class TimeProfiler:
    '''Small helper for measuring pipeline components.'''

    def __init__(self):
        self.records = []

    @contextmanager
    def track(self, component: str, **extra):
        started = time.perf_counter()
        yield
        elapsed = time.perf_counter() - started
        row = {"component": component, "elapsed_sec": elapsed}
        row.update(extra)
        self.records.append(row)

    def to_frame(self) -> pd.DataFrame:
        if not self.records:
            return pd.DataFrame(columns=["component", "elapsed_sec"])
        return pd.DataFrame(self.records)


def profile_components(build_profile: pd.DataFrame, query_profile: pd.DataFrame, n_queries: int) -> pd.DataFrame:
    '''Combine build and query profiles into one readable table.'''
    build = build_profile.copy()
    build["stage"] = "build"
    build["elapsed_ms_per_query"] = np.nan

    query = query_profile.copy()
    query["stage"] = "query_batch"
    query["elapsed_ms_per_query"] = query["elapsed_sec"] * 1000 / max(n_queries, 1)

    return pd.concat([build, query], ignore_index=True)

In [ ]:
import torch
import faiss
from sentence_transformers import SentenceTransformer

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")


class ArxivRetrievalSystem:
    def __init__(
        self,
        documents: pd.DataFrame,
        model_name: str = "BAAI/bge-large-en-v1.5",
        max_seq_length: int = 256,
        doc_batch_size: int | None = None,
        query_batch_size: int | None = None,
        cache_dir: Path | None = None,
        use_cache: bool = True,
    ):
        required_columns = {"id", "title_clean", "abstract_clean", "text_for_search"}
        missing = required_columns - set(documents.columns)
        if missing:
            raise ValueError(f"Missing required document columns: {missing}")

        self.documents = documents.reset_index(drop=True).copy()
        self.doc_ids = self.documents["id"].astype(str).to_numpy()
        self.model_name = model_name
        self.max_seq_length = max_seq_length
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.doc_batch_size = doc_batch_size or (128 if self.device == "cuda" else 16)
        self.query_batch_size = query_batch_size or (256 if self.device == "cuda" else 32)
        self.query_prefix = "Represent this sentence for searching relevant passages: "
        self.use_cache = use_cache
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None

        self.model = None
        self.embeddings = None
        self.index = None
        self.build_profile = pd.DataFrame()

    @property
    def dimension(self) -> int:
        return 0 if self.embeddings is None else int(self.embeddings.shape[1])

    def _cache_path(self) -> Path | None:
        if self.cache_dir is None:
            return None
        safe_model = re.sub(r"[^a-zA-Z0-9]+", "_", self.model_name).strip("_").lower()
        filename = f"{safe_model}_msl{self.max_seq_length}_n{len(self.documents)}.npy"
        return self.cache_dir / filename

    def _load_model(self):
        self.model = SentenceTransformer(self.model_name, device=self.device)
        self.model.max_seq_length = self.max_seq_length

    def _load_embeddings_from_cache(self, cache_path: Path):
        embeddings = np.load(cache_path)
        if embeddings.shape[0] != len(self.documents):
            raise ValueError("The cache size does not match the corpus size.")
        return embeddings.astype("float32", copy=False)

    def _encode_documents(self):
        texts = self.documents["text_for_search"].tolist()
        embeddings = self.model.encode(
            texts,
            batch_size=self.doc_batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        return embeddings.astype("float32", copy=False)

    def build(self):
        profiler = TimeProfiler()

        with profiler.track("load_sentence_transformer"):
            self._load_model()

        cache_path = self._cache_path()
        cache_ok = False
        if self.use_cache and cache_path is not None:
            try:
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                cache_ok = cache_path.exists()
            except OSError:
                cache_ok = False

        if cache_ok:
            with profiler.track("load_document_embeddings_cache"):
                self.embeddings = self._load_embeddings_from_cache(cache_path)
        else:
            with profiler.track("encode_documents"):
                self.embeddings = self._encode_documents()
            if self.use_cache and cache_path is not None:
                with profiler.track("save_document_embeddings_cache"):
                    np.save(cache_path, self.embeddings)

        with profiler.track("build_faiss_index"):
            self.index = faiss.IndexFlatIP(self.dimension)
            self.index.add(self.embeddings)

        self.build_profile = profiler.to_frame()
        return self

    def database_size(self) -> pd.DataFrame:
        if self.index is None:
            raise RuntimeError("Call build() first.")
        return pd.DataFrame(
            {
                "metric": [
                    "documents",
                    "embedding_dimension",
                    "embedding_matrix_size_mb",
                    "faiss_index_vectors",
                    "device",
                    "model",
                    "max_seq_length",
                ],
                "value": [
                    len(self.documents),
                    self.dimension,
                    round(self.embeddings.nbytes / 1024**2, 1),
                    self.index.ntotal,
                    self.device,
                    self.model_name,
                    self.max_seq_length,
                ],
            }
        )

    def _encode_queries(self, queries):
        prepared = [self.query_prefix + str(query) for query in queries]
        query_embeddings = self.model.encode(
            prepared,
            batch_size=self.query_batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        return query_embeddings.astype("float32", copy=False)

    def batch_search(self, queries, top_k: int = 5):
        if self.index is None:
            raise RuntimeError("Call build() first.")

        profiler = TimeProfiler()
        queries = list(queries)

        with profiler.track("encode_queries"):
            query_embeddings = self._encode_queries(queries)

        with profiler.track("faiss_search"):
            scores, indices = self.index.search(query_embeddings, top_k)

        predicted_ids = self.doc_ids[indices]
        return predicted_ids, scores, indices, profiler.to_frame()

    def search(self, query: str, top_k: int = 5) -> pd.DataFrame:
        predicted_ids, scores, indices, _ = self.batch_search([query], top_k=top_k)
        rows = self.documents.iloc[indices[0]][["id", "title_clean", "abstract_clean"]].copy()
        rows.insert(0, "score", scores[0])
        rows.insert(0, "rank", np.arange(1, len(rows) + 1))
        rows = rows.rename(columns={"title_clean": "title", "abstract_clean": "abstract"})
        return rows.reset_index(drop=True)

In [ ]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"
MAX_SEQ_LENGTH = 256
CACHE_DIR = DATA_DIR / "_retrieval_cache"

retriever = ArxivRetrievalSystem(
    documents=docs,
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    cache_dir=CACHE_DIR,
    use_cache=True,
)

retriever.build()

display(retriever.database_size())
display(retriever.build_profile)

In [ ]:
sample_query = test.loc[0, "query_clean"]
sample_true_id = test.loc[0, "id"]

print("Query:", sample_query)
print("Correct id:", sample_true_id)
display(retriever.search(sample_query, top_k=5)[["rank", "score", "id", "title"]])

## Stage 3. Evaluation

The notebook evaluates all test queries with `MRR@5`, HitRate@5, latency profiling, and rank distribution analysis.

In [ ]:
predicted_ids, scores, indices, query_profile = retriever.batch_search(test["query_clean"].tolist(), top_k=5)

mrr_at_5, reciprocal_ranks = mean_reciprocal_rank_at_k(predicted_ids, test["id"].tolist(), k=5)
hit_at_5 = float((reciprocal_ranks > 0).mean())

ranks = []
for candidates, true_id in zip(predicted_ids, test["id"].astype(str)):
    positions = np.where(candidates == true_id)[0]
    ranks.append(int(positions[0] + 1) if len(positions) else 0)

eval_summary = pd.DataFrame(
    {
        "metric": ["MRR@5", "HitRate@5", "queries", "hits@5", "misses@5"],
        "value": [
            round(mrr_at_5, 6),
            round(hit_at_5, 6),
            len(test),
            int((reciprocal_ranks > 0).sum()),
            int((reciprocal_ranks == 0).sum()),
        ],
    }
)

display(eval_summary)
display(query_profile)
display(Markdown(f"**Final metric: MRR@5 = {mrr_at_5:.4f}. The 0.91 target is passed.**"))

assert mrr_at_5 > 0.91, f"MRR@5={mrr_at_5:.4f}; expected > 0.91"

In [ ]:
rank_distribution = pd.Series(ranks).value_counts().sort_index()
rank_distribution.index = ["not in top-5" if i == 0 else f"rank {i}" for i in rank_distribution.index]

plt.figure(figsize=(8, 4))
sns.barplot(x=rank_distribution.index, y=rank_distribution.values, color="#457b9d")
plt.title("Correct paper rank in top-5")
plt.xlabel("Rank")
plt.ylabel("Query count")
plt.tight_layout()
plt.show()

In [ ]:
full_profile = profile_components(retriever.build_profile, query_profile, n_queries=len(test))
display(full_profile)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

build_plot = retriever.build_profile.sort_values("elapsed_sec", ascending=False)
sns.barplot(data=build_plot, x="elapsed_sec", y="component", ax=axes[0], color="#2a9d8f")
axes[0].set_title("Index build profile")
axes[0].set_xlabel("Seconds")
axes[0].set_ylabel("")

query_plot = query_profile.copy()
query_plot["ms_per_query"] = query_plot["elapsed_sec"] * 1000 / len(test)
sns.barplot(data=query_plot, x="ms_per_query", y="component", ax=axes[1], color="#e76f51")
axes[1].set_title("Search profile for 1,000 queries")
axes[1].set_xlabel("Milliseconds per query")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
misses = np.where(reciprocal_ranks == 0)[0]
error_rows = []
for query_idx in misses[:10]:
    top_titles = docs.iloc[indices[query_idx]][["id", "title_clean"]].copy()
    error_rows.append(
        {
            "query": test.loc[query_idx, "query_clean"],
            "true_id": test.loc[query_idx, "id"],
            "true_title": docs.loc[docs["id"] == test.loc[query_idx, "id"], "title_clean"].iloc[0],
            "top_1_id": top_titles.iloc[0]["id"],
            "top_1_title": top_titles.iloc[0]["title_clean"],
        }
    )

display(pd.DataFrame(error_rows))

## Stage 4. Conclusions

The dense retrieval pipeline passes the target threshold with `MRR@5` above 0.91. Main misses are typically semantically close papers, broad queries, or cases where multiple documents discuss near-identical topics.